# Graph Partitioning

## Learning Objectives

1. **Define** min-cut and balanced graph partitioning
2. **Implement** the Kernighan-Lin local search algorithm
3. **Explain** why balanced partitioning is NP-hard while min-cut is polynomial
4. **Describe** the METIS multilevel approach

## Min-Cut vs Balanced Partitioning

**Min-cut:** partition $V = S \cup \bar{S}$ to minimize $\text{cut}(S, \bar{S}) = |\{(u,v)\in E : u\in S, v\in\bar{S}\}|$.

- Solvable in polynomial time via max-flow (Ford-Fulkerson) in $O(mn)$
- Problem: trivial solution is $S = \{v\}$ for any single node (often cut = degree(v))

**Balanced partitioning:** minimize cut subject to $|S| \approx |\bar{S}|$.

- NP-hard in general
- Approximation: spectral methods achieve $O(\sqrt{\log n})$-approximation for balanced bisection

**Ratio cut:** $\frac{\text{cut}(S,\bar{S})}{|S| \cdot |\bar{S}|}$ — penalizes unbalanced cuts; NP-hard to minimize exactly.

In [1]:
from collections import defaultdict

def cut_size(adj, S):
    S = set(S)
    return sum(1 for u in S for v in adj[u] if v not in S)

def kernighan_lin_pass(adj, S, T):
    """
    One pass of Kernighan-Lin: find the best sequence of swaps.
    S, T: two equal-size sets of nodes (mutable copies).
    Returns best cumulative gain and the sequence of swaps to reach it.
    """
    S, T = set(S), set(T)
    unlocked_S, unlocked_T = set(S), set(T)
    gains = []
    swap_seq = []
    total_gain = 0

    for _ in range(min(len(unlocked_S), len(unlocked_T))):
        best_gain = float('-inf'); best_swap = None
        for a in unlocked_S:
            for b in unlocked_T:
                # gain of swapping a and b
                D_a = sum(1 for v in adj[a] if v in T) - sum(1 for v in adj[a] if v in S and v!=a)
                D_b = sum(1 for v in adj[b] if v in S) - sum(1 for v in adj[b] if v in T and v!=b)
                c_ab = 1 if b in adj[a] else 0
                gain = D_a + D_b - 2*c_ab
                if gain > best_gain:
                    best_gain = gain; best_swap = (a, b)
        if best_swap is None: break
        a, b = best_swap
        S.discard(a); S.add(b)
        T.discard(b); T.add(a)
        unlocked_S.discard(a); unlocked_T.discard(b)
        gains.append(best_gain); swap_seq.append((a,b))
        total_gain += best_gain

    # Find prefix with maximum cumulative gain
    best_prefix = 0; cum = 0; best_cum = 0
    for i, g in enumerate(gains):
        cum += g
        if cum > best_cum: best_cum = cum; best_prefix = i+1
    return best_cum, swap_seq[:best_prefix]

import random
rng = random.Random(42)
adj = defaultdict(set)
# Two communities
for i in range(8):
    for j in range(i+1,8):
        if i<4 and j<4 and rng.random()<0.8: adj[i].add(j); adj[j].add(i)
        elif i>=4 and j>=4 and rng.random()<0.8: adj[i].add(j); adj[j].add(i)
        elif rng.random()<0.1: adj[i].add(j); adj[j].add(i)

S = set(range(0,4)); T = set(range(4,8))
print(f"Initial cut: {cut_size(adj, S)}")
gain, swaps = kernighan_lin_pass(adj, S, T)
print(f"Best gain from one KL pass: {gain}")
print(f"Swaps to apply: {swaps}")

Initial cut: 3
Best gain from one KL pass: 0
Swaps to apply: []


## Kernighan-Lin Algorithm

**Algorithm:**
1. Start with a random balanced partition $(S, T)$ with $|S| = |T|$
2. Repeat until no improvement:
   - Run one KL pass: find the best sequence of node swaps that maximizes gain
   - Apply the best-gain prefix of swaps
3. Return the best partition found

**Gain of swapping $a \in S$ and $b \in T$:**
$$g(a,b) = D(a) + D(b) - 2c(a,b)$$
where $D(v)$ = external degree − internal degree, $c(a,b) = 1$ if $(a,b) \in E$.

**Complexity:** $O(n^2 \log n)$ per pass. Usually converges in 2–4 passes.
**Works well:** starting from a spectral partition, KL refines the cut significantly.

In [2]:
def kernighan_lin(adj, n_passes=5, seed=42):
    """Kernighan-Lin graph bisection. Returns (cut_size, S, T)."""
    rng2 = random.Random(seed)
    nodes = list(adj.keys())
    if len(nodes) % 2 == 1: nodes = nodes[:-1]
    rng2.shuffle(nodes)
    S = set(nodes[:len(nodes)//2])
    T = set(nodes[len(nodes)//2:])
    best_cut = cut_size(adj, S)
    for _ in range(n_passes):
        gain, swaps = kernighan_lin_pass(adj, S, T)
        if gain <= 0: break
        for a, b in swaps:
            S.discard(a); S.add(b)
            T.discard(b); T.add(a)
        best_cut = cut_size(adj, S)
    return best_cut, S, T

cut, S_final, T_final = kernighan_lin(adj)
print(f"KL final cut: {cut}")
print(f"S = {sorted(S_final)}")
print(f"T = {sorted(T_final)}")

KL final cut: 3
S = [4, 5, 6, 7]
T = [0, 1, 2, 3]


## METIS: Multilevel Partitioning

METIS (Karypis & Kumar 1998) is the state-of-the-art graph partitioning tool:

**Phase 1 — Coarsening:**
Repeatedly match and collapse pairs of nodes until graph has ~100 nodes.
Use **heavy-edge matching:** match each node with its highest-weight unmatched neighbor.

**Phase 2 — Initial partitioning:**
Partition the small coarse graph using KL or spectral methods.

**Phase 3 — Uncoarsening + refinement:**
Expand back to original graph level by level.
Apply KL refinement at each level to fix errors introduced by coarsening.

**Complexity:** near-linear $O(m)$.
**Quality:** within 5% of optimal on typical graphs.

## Summary

| Method | Complexity | Balance | Quality |
|--------|-----------|---------|---------|
| Min-cut (max-flow) | O(mn) | No | Exact |
| Spectral bisection | O(n²–n³) | Yes | ≈ √logn approx |
| Kernighan-Lin | O(n² log n) | Yes | Local optimum |
| METIS | O(m) | Yes | Near-optimal |

Use **KL** for small graphs or as post-processing; **METIS** for large graphs.
Graph partitioning is critical for parallel computing — partitioning a computation graph minimizes inter-processor communication.